Сначала код группирует процессоры по шести факторам, которые должны совпадать точно. 
Затем ищет внутри этих групп 20 строк с минимальным разбросом по частоте и возрасту. 
Чтобы сравнить ГГц и годы, каждый размах делится на стандартное отклонение своего признака по всему набору без пропусков. 
При отборе cpu_mark не используется.

In [1]:
import pandas as pd

df = pd.read_csv("final5.csv")
data = df.loc[df["specs_missing_count"].eq(0)]

fixed = [
    "manufacturer_Intel", "manufacturer_Other", "category_Laptop",
    "cores", "tdp_w", "l3_cache_mb",
]
clock, age = "base_clock_ghz", "years_ago"
size = 20
clock_scale, age_scale = data[[clock, age]].std()

best_score = float("inf")
extra = None

for _, group in data.groupby(fixed):
    if len(group) < size:
        continue

    clocks = sorted(group[clock].unique())
    for i, low in enumerate(clocks):
        for high in clocks[i:]:
            window = group.loc[group[clock].between(low, high)].sort_values(age)
            for start in range(len(window) - size + 1):
                candidate = window.iloc[start:start + size]
                clock_span = (candidate[clock].max() - candidate[clock].min()) / clock_scale
                age_span = (candidate[age].max() - candidate[age].min()) / age_scale
                score = max(clock_span, age_span)

                if score < best_score:
                    best_score = score
                    extra = candidate

if extra is None:
    print(f"Группа из {size} строк с совпадающими факторами не найдена.")
else:
    extra.sort_index().to_csv("cpu_extra_sample.csv", index=False)
    print(f"Сохранено {len(extra)} строк в cpu_extra_sample.csv")
    display(extra[fixed + [clock, age]].agg(["min", "max"]))

Сохранено 20 строк в cpu_extra_sample.csv


,manufacturer_Intel,manufacturer_Other,category_Laptop,cores,tdp_w,l3_cache_mb,base_clock_ghz,years_ago
min,1,0,1,2,35.0,3.0,2.1,15
max,1,0,1,2,35.0,3.0,2.5,16


In [2]:
import numpy as np
import pandas as pd

extra = pd.read_csv("cpu_extra_sample.csv")
y = np.log(extra["cpu_mark"])

df_0 = len(y) - 1
S0_sq = ((y - y.mean()) ** 2).sum() / df_0

print(f"Число процессоров = {len(y)}")
print(f"Степени свободы = {df_0}")
print(f"S₀² = {S0_sq:.6f}")

Число процессоров = 20
Степени свободы = 19
S₀² = 0.080688
